# Memory dài hạn cho trợ lý: ghi gì, đọc gì, quên gì

Người dùng kể với trợ lý qua 6 phiên (mỗi tuần một phiên, 108 lượt, phần lớn là chuyện phiếm): tên, dị ứng, con chó,
rồi chuyển nhà, đổi việc, bỏ cà phê. Tới phiên thứ bảy họ hỏi lại. Bài này dựng **memory dài hạn** cho trợ lý: một kho
ghi nhớ có embedding, **chính sách ghi** (lưu nguyên lượt hay trích thông tin bằng LLM, gộp bản trùng), **chính sách
đọc** (điểm liên quan, độ mới, độ quan trọng), **quên** khi kho đầy, và đo nó trên 20 câu hỏi có đáp án.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` gọi Ollama (`qwen2.5:3b`, `bge-m3`), in bảng; test bỏ qua chúng.
- Chấm: `uv run pytest tracks/ai-engineer/modules/16-ai-agents/exercises/intermediate/03-agent-memory -q`

In [ ]:
# explore: đưa gốc repo vào sys.path để import llmkit (test chạy từ gốc repo nên không cần cell này)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import time
from dataclasses import dataclass, field
from typing import Callable

import numpy as np
import pandas as pd

import dialogue

## 1. Kho ghi nhớ

Mỗi ghi nhớ là một `Memory` (văn bản, thời điểm tính bằng giờ, độ quan trọng trong `[0, 1]`, nguồn, vector đã chuẩn
hoá). `embed` là hàm nhận list chuỗi, trả mảng `(n, d)` (ví dụ `llmkit.embed`, hoặc `FakeLLM().embed` trong test);
`MemoryStore` tự chuẩn hoá L2 mỗi vector.

`recency_weight(age_hours, half_life_hours)` $$= 0.5^{\max(age, 0) / h}$$: ghi nhớ đúng `half_life` giờ tuổi có trọng số
0.5 (mục 5 lý thuyết).

`MemoryStore(embed, dedup_threshold=0.95, capacity=None, half_life_hours=168)`:

- `add(text, t, importance=0.5, source=None) -> str`:
  - văn bản rỗng (sau `strip`) thì không làm gì, trả `"skipped"`;
  - có ghi nhớ cũ với cosine **lớn hơn hoặc bằng** `dedup_threshold` (lấy cái cao nhất): **cập nhật** nó thành văn bản,
    thời điểm, nguồn, vector mới, `importance` lấy giá trị lớn hơn; giữ `id` cũ; trả `"updated"`;
  - ngược lại thêm `Memory` mới với `id` tăng dần từ 0, trả `"added"`; nếu vượt `capacity` thì **xoá** một ghi nhớ
    **khác** cái vừa thêm có `importance * recency_weight(t - m.t)` nhỏ nhất (bằng nhau thì cái cũ hơn, rồi `id` nhỏ hơn).
- `search(query, now, k=5, w_rel=1.0, w_rec=0.0, w_imp=0.0) -> list[(Memory, score)]`:
  `score = w_rel * cos + w_rec * recency_weight(now - m.t) + w_imp * importance`, `k` cái điểm cao nhất, giảm dần;
  bằng điểm thì ghi nhớ **mới hơn** đứng trước, rồi `id` lớn hơn. Kho rỗng trả `[]`.

In [ ]:
def recency_weight(age_hours: float, half_life_hours: float) -> float:
    return 0.5 ** (max(age_hours, 0.0) / half_life_hours)


@dataclass
class Memory:
    id: int
    text: str
    t: float
    importance: float
    source: int | None
    vector: np.ndarray = field(repr=False)


class MemoryStore:
    def __init__(self, embed: Callable[[list[str]], np.ndarray], *, dedup_threshold: float = 0.95,
                 capacity: int | None = None, half_life_hours: float = 24 * 7):
        self.embed, self.dedup, self.capacity, self.half_life = embed, dedup_threshold, capacity, half_life_hours
        self.items: list[Memory] = []
        self._next_id = 0

    def _vector(self, text: str) -> np.ndarray:
        v = np.asarray(self.embed([text])[0], dtype=np.float32)
        return v / np.linalg.norm(v)

    def add(self, text: str, t: float, importance: float = 0.5, source: int | None = None) -> str:
        text = text.strip()
        if not text:
            return "skipped"
        v = self._vector(text)
        if self.items:
            sims = np.stack([m.vector for m in self.items]) @ v
            best = int(np.argmax(sims))
            if sims[best] >= self.dedup:
                m = self.items[best]
                m.text, m.t, m.source, m.vector = text, t, source, v
                m.importance = max(m.importance, importance)
                return "updated"
        self.items.append(Memory(self._next_id, text, t, importance, source, v))
        self._next_id += 1
        if self.capacity is not None and len(self.items) > self.capacity:
            victim = min(self.items[:-1], key=lambda m: (m.importance * recency_weight(t - m.t, self.half_life),
                                                         m.t, m.id))
            self.items.remove(victim)
        return "added"

    def search(self, query: str, now: float, k: int = 5, w_rel: float = 1.0, w_rec: float = 0.0,
               w_imp: float = 0.0) -> list[tuple[Memory, float]]:
        if not self.items:
            return []
        sims = np.stack([m.vector for m in self.items]) @ self._vector(query)
        scored = [(m, float(w_rel * s + w_rec * recency_weight(now - m.t, self.half_life) + w_imp * m.importance))
                  for m, s in zip(self.items, sims)]
        scored.sort(key=lambda pair: (-pair[1], -pair[0].t, -pair[0].id))
        return scored[:k]

## 2. Chính sách ghi: trích thông tin bằng LLM

Lưu nguyên mọi lượt người dùng nói là cách đơn giản nhất, nhưng 84/108 lượt ở đây là chuyện phiếm. Cách thứ hai: nhờ
LLM **trích** các thông tin lâu dài về người dùng thành câu ngắn ("Người dùng bị dị ứng hải sản."), lượt không có gì
thì không ghi.

`extract_facts(llm, user_turn) -> list[str]`: gọi `llm.chat` **một** lần với hai message: `system` là
`EXTRACT_SYSTEM`, `user` là `f'Câu: "{user_turn}"'`, và `json_schema=FACT_SCHEMA`. Trả list chuỗi đã `strip`, bỏ
phần tử rỗng hoặc không phải chuỗi. JSON hỏng, thiếu khoá `facts`, `facts` không phải list, hoặc `llm.chat` ném
`llmkit.LLMError` thì trả `[]`.

In [ ]:
import llmkit

FACT_SCHEMA = {"type": "object", "required": ["facts"],
               "properties": {"facts": {"type": "array", "items": {"type": "string"}}}}
EXTRACT_SYSTEM = (
    "Bạn ghi nhớ thông tin về người dùng. Đọc câu người dùng nói và liệt kê các thông tin về bản thân người dùng "
    "(tên, tuổi, nơi sống, công việc, người thân, thú cưng, sở thích, thói quen, sức khoẻ, kế hoạch, đồ dùng). "
    "Mỗi thông tin là một câu ngắn bắt đầu bằng \"Người dùng\". Nếu câu chỉ là câu hỏi hoặc nhờ việc, trả danh sách rỗng.\n\n"
    "Ví dụ:\nCâu: \"Mình làm y tá ở Cần Thơ, có hai con mèo.\"\n"
    "{\"facts\": [\"Người dùng làm y tá.\", \"Người dùng sống ở Cần Thơ.\", \"Người dùng có hai con mèo.\"]}\n"
    "Câu: \"Dịch giúp mình chữ 'hello'.\"\n{\"facts\": []}")


def extract_facts(llm, user_turn: str) -> list[str]:
    messages = [{"role": "system", "content": EXTRACT_SYSTEM}, {"role": "user", "content": f'Câu: "{user_turn}"'}]
    try:
        facts = json.loads(llm.chat(messages, json_schema=FACT_SCHEMA))["facts"]
    except (llmkit.LLMError, ValueError, KeyError, TypeError):
        return []
    if not isinstance(facts, list):
        return []
    return [f.strip() for f in facts if isinstance(f, str) and f.strip()]

## 3. Chạy thật

Thời điểm: phiên $$s$$ ở giờ $$168 s$$ (mỗi tuần một phiên), lượt thứ $$i$$ cộng thêm $$0.05 i$$ giờ; câu hỏi được hỏi ở
giờ $$1008$$ (đầu tuần thứ bảy). Hai kho cùng `dedup_threshold=0.95`:

- **raw**: mỗi lượt người dùng là một ghi nhớ, nguồn là `fact_id` của lượt đó (`None` với chuyện phiếm).
- **fact**: chỉ các câu `extract_facts` trích được, nguồn là `fact_id` của lượt sinh ra nó.

Rồi đo **retrieval** (hit@k: trong top-k có ghi nhớ đúng nguồn không, 18 câu có bằng chứng) và **trả lời** (20 câu, chấm
bằng từ khoá) với bốn điều kiện: không có memory, nhét **toàn bộ** lịch sử vào context, memory raw top-5, memory fact
top-5. Khoảng 200 lượt gọi model. Điều kiện "toàn bộ lịch sử" chậm hẳn các điều kiện khác (prompt vượt context mặc
định 4 096 token của Ollama); cả notebook chạy khoảng 12 phút trên PC của track.

In [ ]:
# explore: dựng hai kho ghi nhớ (khoảng 2 tới 3 phút: 108 lượt gọi LLM trích thông tin + embed)
from llmkit import get_client

turns = dialogue.build_sessions()
NOW = 6 * 168.0


def t_of(turn):
    return turn["session"] * 168.0 + turn["idx"] * 0.05


llm = get_client()
raw = MemoryStore(llmkit.embed)
facts = MemoryStore(llmkit.embed)
written = {"raw": {}, "fact": {}}
extracted = []
start = time.perf_counter()
for turn in turns:
    status = raw.add(turn["user"], t_of(turn), source=turn["fact_id"])
    written["raw"][status] = written["raw"].get(status, 0) + 1
    found = extract_facts(llm, turn["user"])
    extracted.append((turn, found))
    for f in found:
        status = facts.add(f, t_of(turn), source=turn["fact_id"])
        written["fact"][status] = written["fact"].get(status, 0) + 1
print(f"{len(turns)} lượt, {sum(t['fact_id'] is not None for t in turns)} lượt có thông tin, "
      f"{time.perf_counter() - start:.0f} s")
print("ghi:", written, "| số ghi nhớ:", len(raw.items), len(facts.items))
fp = [(t["user"], f) for t, f in extracted if t["fact_id"] is None and f]
fn = [t["user"] for t, f in extracted if t["fact_id"] is not None and not f]
print(f"chuyện phiếm mà vẫn trích ra thông tin: {len(fp)} | lượt có thông tin mà trích rỗng: {len(fn)}")
for user, f in fp[:5]:
    print("   ", user, "->", f)
for t, f in extracted:
    if t["fact_id"] in (3, 17, 12):
        print("   ", t["user"], "->", f)

In [ ]:
# explore: retrieval hit@k, có và không có trọng số độ mới
def hits(store, k, w_rec):
    out = {}
    for question, keywords, gold, group in dialogue.QUESTIONS:
        if not gold:
            continue
        found = store.search(question, NOW, k=k, w_rec=w_rec)
        out.setdefault(group, []).append(any(m.source in gold for m, _ in found))
    row = {g: f"{sum(v)}/{len(v)}" for g, v in out.items()}
    row["tổng"] = f"{sum(sum(v) for v in out.values())}/{sum(len(v) for v in out.values())}"
    return row


table = []
for name, store in [("raw", raw), ("fact", facts)]:
    for k in (1, 3, 5):
        for w_rec in (0.0, 0.3):
            table.append({"kho": name, "k": k, "w_rec": w_rec, **hits(store, k, w_rec)})
print(pd.DataFrame(table).to_string(index=False))

In [ ]:
# explore: quên khi kho đầy: kho fact giới hạn 20 ghi nhớ (importance như nhau, chỉ còn độ mới quyết định)
small = MemoryStore(llmkit.embed, capacity=20)
for turn, found in extracted:
    for f in found:
        small.add(f, t_of(turn), source=turn["fact_id"])
print("capacity 20:", hits(small, 5, 0.0), "| không giới hạn:", hits(facts, 5, 0.0))

In [ ]:
# explore: trả lời 20 câu với 4 điều kiện (khoảng 10 phút, phần lớn là điều kiện toàn bộ lịch sử)
import httpx

SYSTEM = ("Bạn là trợ lý cá nhân của người dùng. Trả lời ngắn gọn, tiếng Việt. Chỉ dùng thông tin người dùng đã kể; "
          "nếu không có thông tin thì nói không biết.")


def ask(question, condition):
    messages = [{"role": "system", "content": SYSTEM}]
    if condition == "history":
        for turn in turns:
            messages += [{"role": "user", "content": turn["user"]}, {"role": "assistant", "content": turn["assistant"]}]
    elif condition in ("mem_raw", "mem_fact"):
        store = raw if condition == "mem_raw" else facts
        notes = "\n".join(f"- {m.text}" for m, _ in store.search(question, NOW, k=5, w_rec=0.3))
        messages[0]["content"] += "\n\nGhi nhớ về người dùng (mới nhất ưu tiên):\n" + notes
    messages.append({"role": "user", "content": question})
    r = httpx.post(os.environ.get("OLLAMA_HOST", "http://localhost:11434") + "/api/chat", timeout=600, json={
        "model": "qwen2.5:3b", "messages": messages, "stream": False, "options": {"temperature": 0, "seed": 0}}).json()
    return r["message"]["content"], r.get("prompt_eval_count")


rows = []
for condition in ["none", "history", "mem_raw", "mem_fact"]:
    for question, keywords, gold, group in dialogue.QUESTIONS:
        start = time.perf_counter()
        answer, prompt_tokens = ask(question, condition)
        rows.append(dict(condition=condition, group=group, question=question, answer=answer,
                         ok=any(k in answer.lower() for k in keywords), prompt_tokens=prompt_tokens,
                         seconds=time.perf_counter() - start))
        print(f"{condition:8s} {'ĐÚNG' if rows[-1]['ok'] else 'sai '} {group:9s} {question} -> {answer[:70]!r}")
qa = pd.DataFrame(rows)
print(qa.pivot_table(index="group", columns="condition", values="ok", aggfunc="sum", sort=False))
print(qa.groupby("condition", sort=False).agg(dung=("ok", "sum"), token_prompt=("prompt_tokens", "mean"),
                                              giay_trung_vi=("seconds", "median")).round(1))

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Nhét cả lịch sử vào context có "nhớ" được phiên đầu không? In `prompt_tokens` của điều kiện đó và so với số token
  thật của toàn bộ lịch sử (`approx_tokens` của bài Basic 03): phần nào đã bị Ollama bỏ đi?
- Trọng số độ mới `w_rec` giúp nhóm câu hỏi nào, làm hại nhóm nào? Vì sao với thông tin cũ không đổi (tên, dị ứng)
  độ mới lại là nhiễu?
- Đọc các câu chuyện phiếm mà LLM vẫn trích ra "thông tin", và câu "Mực vừa được 3 tuổi hôm qua.": trích từng lượt
  một, không có ngữ cảnh, mất gì?
- Câu hỏi về thông tin **không có** ("Mình có nuôi mèo không?"): chấm bằng từ khoá "không" có đáng tin không?

Sau đó làm phần trắc nghiệm của bài trong app.